#### Imports

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset
from datasets import load_dataset
import re
import torch
from transformers import pipeline, AutoTokenizer

import matplotlib.pyplot as plt
import numpy as np
import os
import json
from datetime import datetime

2026-05-11 05:52:50.376865: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778478770.429757    3211 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778478770.445437    3211 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778478770.539942    3211 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778478770.539966    3211 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778478770.539968    3211 computation_placer.cc:177] computation placer alr

#### Data Retrieval

In [2]:
seed = 42

# Load UltraFeedback Dataset
ds = load_dataset("openbmb/UltraFeedback")

print(ds)

README.md: 0.00B [00:00, ?B/s]

evol_instruct.jsonl:   0%|          | 0.00/168M [00:00<?, ?B/s]

false_qa.jsonl:   0%|          | 0.00/25.9M [00:00<?, ?B/s]

flan.jsonl:   0%|          | 0.00/240M [00:00<?, ?B/s]

sharegpt.jsonl:   0%|          | 0.00/313M [00:00<?, ?B/s]

truthful_qa.jsonl: 0.00B [00:00, ?B/s]

ultrachat.jsonl:   0%|          | 0.00/182M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/63967 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['source', 'instruction', 'models', 'completions', 'correct_answers', 'incorrect_answers'],
        num_rows: 63967
    })
})


In [3]:
print(ds["train"][3]['instruction'])

How can I convert the decimal number 31 to binary format using JavaScript code? Can you provide the code for this conversion?


#### GRM Initial Setup

In [4]:
# Basic GRM Reward Pipeline
pipe = pipeline(
    "text-generation",
    tokenizer=AutoTokenizer.from_pretrained("BBQGOD/DeepSeek-GRM-16B", trust_remote_code=True),
    model="BBQGOD/DeepSeek-GRM-16B",
    model_kwargs={"torch_dtype": torch.bfloat16},
    device="cuda",  # replace with "mps" to run on a Mac device
    trust_remote_code=True
)

def extract_last_floats(text: str) -> list[float]:
    pattern = re.compile(
        r'(?:\\{1,2}boxed\{|\[)'
        r'\s*([^\]\}]+?)\s*'
        r'(?:\}|\])'
    )
    matches = list(pattern.finditer(text))
    if not matches:
        return []
    last_content = matches[-1].group(1)
    parts = re.split(r'\s*,\s*', last_content.strip())
    floats = []
    for p in parts:
        try:
            floats.append(float(p))
        except ValueError:
            pass
    return floats


messages = [{
    "role": "user",
    "content": "You are a skilled little expert at scoring responses. You should evaluate given responses based on the given judging criteria.\nGiven the context of the conversation (the last round is the User's query) and multiple responses from the Assistant, you need to refer to the [General Evaluation Criteria] to score the responses. Based on the general evaluation criteria, state potential other specific criteria to the query, the weights of different criteria, and then provide an overall comprehensive score upon them.\nEach score is an integer between 1 and 10, with a higher score indicating that the response meets the relevant criteria more closely. For example, a score of 1 means the response does not meet the criteria at all, a score of 6 means the response meets only some parts, and a score of 10 means the response perfectly meets the evaluation criteria.\nBefore scoring, please analyze step by step. Your scoring needs to be as strict as possible.\n#### Evaluation Criteria ####\n1. Instruction Adherence:\n   - Fully Adhered (9-10 points): The response fully complies with all instructions and requirements of the question.\n   - Partially Adhered (6-8 points): The response meets most of the instructions but has some omissions or misunderstandings.\n   - Basically Adhered (3-5 points): The response meets some instructions, but the main requirements are not fulfilled.\n   - Not Adhered (1-2 points): The response does not meet any instructions.\n   Example: If the question requires three examples and the response provides only one, it falls under \"Partially Adhered.\"\n2. Usefulness:\n   - Highly Useful (9-10 points): The response provides comprehensive and accurate information, fully addressing the issue.\n   - Useful but Incomplete (6-8 points): The response provides some useful information, but lacks details or accuracy.\n   - Limited Usefulness (3-5 points): The response offers little useful information, with most content being irrelevant or incorrect.\n   - Useless or Incorrect (1-2 points): The response is completely irrelevant or incorrect.\n   Example: If there are factual errors in the response but the overall direction is correct, it falls under \"Useful but Incomplete.\"\n3. Level of Detail:\n   - Very Detailed (9-10 points): The response includes ample details covering all aspects of the issue.\n   - Detailed but Slightly Lacking (6-8 points): The response is fairly detailed but misses some important details.\n   - Basically Detailed (3-5 points): The response provides some details but is not thorough enough overall.\n   - Not Detailed (1-2 points): The response is very brief and lacks necessary details.\n   Example: If the response provides only a simple conclusion without an explanation, it falls under \"Not Detailed.\"\n4. Relevance:\n   - Highly Relevant (9-10 points): The response is highly relevant to the question, with information closely aligned with the topic.\n   - Generally Relevant (6-8 points): The response is generally relevant but includes some unnecessary information.\n   - Partially Relevant (3-5 points): The response has a lot of content that deviates from the topic.\n   - Not Relevant (1-2 points): The response is completely irrelevant.\n   Example: If the response strays from the topic but still provides some relevant information, it falls under \"Partially Relevant.\"\n\n#### Conversation Context ####\n{}\n#### Responses to be Scored ####\n{}\n\n#### Output Format Requirements ####\n\nOutput with three lines\nSpecific Criteria: <Other potential criteria specific to the query and the context, and the weights of each criteria>.\nAnalysis: <Compare different responses based on given Criteria>.\nScores: <the overall comprehensive score of all resposnes in order, seperate by comma in the boxed, e.g., \\boxed{{x, x}} if there exists 2 responeses>."
}]

single_turn_query = "You are an expert in the uses and creation of spreadsheets in Microsoft Excel. Create a formula that will look for the same entry in cell CB5 from columns D and E and then return all entries from the corresponding rows of these entries from Q5 to T5 and return the last 6 entries in Q5 to T500 by taking the date from cell CK4 and looking up column B"

single_turn_responses = [
    "Unfortunately I do not have access to the spreadsheet data you referenced, but I can provide an example formula that attempts to accomplish the described logic:\n\n```\n=IF(OR(D5=CB5,E5=CB5),INDEX(Q5:T5,MATCH(CB5,{D5:E5},0)),INDEX(Q5:T500,MATCH(MAX(IF(B5:B500<=CK4,B5:B500)),B5:B500,0)-5))\n```\n\nBreakdown:\n\n- The IF statement checks if CB5 matches either D5 or E5 using OR. \n- If there is a match, it returns the range Q5:T5 for that row using INDEX and MATCH.\n- If no match, it returns the last 6 rows of Q:T before the date in CK4. It does this by finding the MAX date less than CK4 using MAX and IF, then matching that date using MATCH and returning 6 rows above it using INDEX.\n\nLet me know if you need any clarification or have additional requirements!", # Best Response
    "To achieve the desired outcome, you can use the following formula in cell T501:\n```\n=INDEX(OFFSET(OFFSET(Q5:T5,MATCH(MAX(D$5:E$5),D$5:E$5,0)),1,6),1)\n```\nThis formula uses the `INDEX` and `OFFSET` functions to retrieve the last 6 entries from columns Q5 to T5 starting from the row with the maximum value in either column D or E. Here's how the formula works:\n\n1. `MATCH(MAX(D$5:E$5),D$5:E$5,0)`: This returns the row number of the cell with the maximum value in columns D5 to E5.\n2. `OFFSET(OFFSET(Q5:T5,MATCH(MAX(D$5:E$5),D$5:E$5,0)),1,6)`: This retrieves a range of 6 cells starting from the row returned by the above formula in columns Q5 to T5.\n3. `INDEX(OFFSET(OFFSET(Q5:T5,MATCH(MAX(D$5:E$5),D$5:E$5,0)),1,6),1)`: This returns the value in the first row of the range retrieved in step 2.\n\nFinally, to link the date in cell CK4 to the first entry in this range, you can use the following formula in cell T500:\n```\n=IF(T501=CK4,T501, \"\")\n```\nThis formula checks whether the value in cell T501 is the same as the date in cell CK4. If they are the same, it returns the value in cell T501. Otherwise, it returns an empty string.",
    " To create a formula that does the task you described, you can use the following formula in cell B500:\n\n=JOIN(\" \",IF(ISNUMBER(MATCH(CB5,D:E,0)),D5:E5,\"\"))\n\nHere's how the formula works:\n* The JOIN function is used to combine the rows of data from columns D and E where the value in cell CB5 exists.\n* The IF function is used to check if the value in cell CB5 exists in columns D and E. If it does, then the formula returns the values in columns D and E for those rows. If it doesn't, then the formula returns an empty string.\n* The MATCH function is used to find the row number where the value in cell CB5 exists in columns D and E. If the value is found, then the row number is returned as an integer. If the value is not found, then the function returns #N/A.\n* The D5:E5 and E5:E5 ranges are used to specify the cells to look up in columns D and E.\n* The last 6 entries in Q5 to T500 are selected by using the following formula in cell B500:\n=JOIN(\" \",IF(ISNUMBER(MATCH(CB5,D:E,0)),D5:E5, \"\"))\n\nThis formula will look for the same entry in cell CB5 in columns D and E, and then return all entries from the corresponding rows of these columns from Q5 to T5. It will also return the last 6 entries in Q5 to T500 by taking the date from cell CK4 and looking up column B.\nNote: The above formula assumes that the data is sorted in ascending order in columns D and E based on the date in cell CK4. If the data is not sorted, you may need to adjust the formula accordingly.",
]

def assemble_responses(resp_list):
    return "".join([f"[The Begin of Response {i+1}]\n{resp}\n[The End of Response {i+1}]\n\n" for i, resp in enumerate(resp_list)])

messages[0]["content"] = messages[0]["content"].format(single_turn_query, assemble_responses(single_turn_responses))
outputs = pipe(messages, max_new_tokens=2048, temperature=1.0, do_sample=True)
judgement = outputs[0]["generated_text"][-1]["content"].strip()
print(judgement) # DeepSeek-GRM tends to generate direct and fixed set of principles when facing simple queries.
rewards = extract_last_floats(judgement)
print(rewards)

/home/ubuntu/.local/lib/python3.10/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


config.json: 0.00B [00:00, ?B/s]

...

model-00002-of-000004.safetensors:   0%|          | 0.00/8.59G [00:00<?, ?B/s]

model-00003-of-000004.safetensors:   0%|          | 0.00/8.59G [00:00<?, ?B/s]

model-00004-of-000004.safetensors:   0%|          | 0.00/5.64G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Specific Criteria: 
1. Instruction Adherence: 
   - The response should directly address the task of finding an entry in CB5 from columns D and E and returning the corresponding rows from Q5 to T5, as well as the last 6 entries from Q5 to T500 based on the date in CK4.
2. Usefulness: 
   - The response should provide a clear and correct formula that achieves the desired outcome.
3. Level of Detail: 
   - The response should explain the steps and logic behind the formula, including any necessary functions and their uses.
4. Relevance: 
   - The response should stay focused on the task without straying into irrelevant details or alternative solutions.

Analysis:
- **Response 1**: This response attempts to address the task but introduces a more complex solution involving multiple functions and conditions. It provides a detailed breakdown of the formula but includes unnecessary complexity. It adheres to the instruction partially and is somewhat useful but lacks direct clarity and simplicit

#### GRM Principle Parsing Extraction

In [5]:
N = 200
subset = ds["train"].shuffle(seed=42).select(range(N))

# Extract principles from the judgement
def extract_principles(text: str) -> list[str]:
    # 1) Extract block between "Specific Criteria:" and "Analysis:"
    block_match = re.search(r"Specific Criteria:\s*(.*?)\s*Analysis:", text, re.DOTALL)

    results = []

    if block_match:
        block = block_match.group(1)

        # 2) Extract content after "[number]. " up to ":"
        results = re.findall(r"\d+\.\s*(.*?)(?=:)", block)

    return results

# Generate the judgement from each query and response.
def generate_judgement(query: str, responses: list[str]) -> str:
    messages = [{
    "role": "user",
    "content": "You are a skilled little expert at scoring responses. You should evaluate given responses based on the given judging criteria.\nGiven the context of the conversation (the last round is the User's query) and multiple responses from the Assistant, you need to refer to the [General Evaluation Criteria] to score the responses. Based on the general evaluation criteria, state potential other specific criteria to the query, the weights of different criteria, and then provide an overall comprehensive score upon them.\nEach score is an integer between 1 and 10, with a higher score indicating that the response meets the relevant criteria more closely. For example, a score of 1 means the response does not meet the criteria at all, a score of 6 means the response meets only some parts, and a score of 10 means the response perfectly meets the evaluation criteria.\nBefore scoring, please analyze step by step. Your scoring needs to be as strict as possible.\n#### Evaluation Criteria ####\n1. Instruction Adherence:\n   - Fully Adhered (9-10 points): The response fully complies with all instructions and requirements of the question.\n   - Partially Adhered (6-8 points): The response meets most of the instructions but has some omissions or misunderstandings.\n   - Basically Adhered (3-5 points): The response meets some instructions, but the main requirements are not fulfilled.\n   - Not Adhered (1-2 points): The response does not meet any instructions.\n   Example: If the question requires three examples and the response provides only one, it falls under \"Partially Adhered.\"\n2. Usefulness:\n   - Highly Useful (9-10 points): The response provides comprehensive and accurate information, fully addressing the issue.\n   - Useful but Incomplete (6-8 points): The response provides some useful information, but lacks details or accuracy.\n   - Limited Usefulness (3-5 points): The response offers little useful information, with most content being irrelevant or incorrect.\n   - Useless or Incorrect (1-2 points): The response is completely irrelevant or incorrect.\n   Example: If there are factual errors in the response but the overall direction is correct, it falls under \"Useful but Incomplete.\"\n3. Level of Detail:\n   - Very Detailed (9-10 points): The response includes ample details covering all aspects of the issue.\n   - Detailed but Slightly Lacking (6-8 points): The response is fairly detailed but misses some important details.\n   - Basically Detailed (3-5 points): The response provides some details but is not thorough enough overall.\n   - Not Detailed (1-2 points): The response is very brief and lacks necessary details.\n   Example: If the response provides only a simple conclusion without an explanation, it falls under \"Not Detailed.\"\n4. Relevance:\n   - Highly Relevant (9-10 points): The response is highly relevant to the question, with information closely aligned with the topic.\n   - Generally Relevant (6-8 points): The response is generally relevant but includes some unnecessary information.\n   - Partially Relevant (3-5 points): The response has a lot of content that deviates from the topic.\n   - Not Relevant (1-2 points): The response is completely irrelevant.\n   Example: If the response strays from the topic but still provides some relevant information, it falls under \"Partially Relevant.\"\n\n#### Conversation Context ####\n{}\n#### Responses to be Scored ####\n{}\n\n#### Output Format Requirements ####\n\nOutput with three lines\nSpecific Criteria: <Other potential criteria specific to the query and the context, and the weights of each criteria>.\nAnalysis: <Compare different responses based on given Criteria>.\nScores: <the overall comprehensive score of all resposnes in order, seperate by comma in the boxed, e.g., \\boxed{{x, x}} if there exists 2 responeses>."
    }]
    messages[0]["content"] = messages[0]["content"].format(query, assemble_responses(responses))
    outputs = pipe(messages, max_new_tokens=2048, temperature=1.0, do_sample=True)
    judgement = outputs[0]["generated_text"][-1]["content"].strip()
    return judgement

# Collect a list of principles from the training set
all_principles = []
raw_observations = []

for entry in subset:
    query = entry['instruction']
    responses = entry['correct_answers'] + entry['incorrect_answers']
    judgement = generate_judgement(query, responses)
    current_principles = extract_principles(judgement)
    all_principles.extend(current_principles)
    raw_observations.append({
        "query": query,
        "responses": responses,
        "judgement": judgement,
        "principles": current_principles
    })

data = {
    "all_principles": all_principles,
    "raw_observations": raw_observations
}

with open("./experiment_runs/principles.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.10/logging/__init__.py", line 1100, in emit
    msg = self.format(record)
  File "/usr/lib/python3.10/logging/__init__.py", line 943, in format
    return fmt.format(record)
  File "/usr/lib/python3.10/logging/__init__.py", line 678, in format
    record.message = record.getMessage()
  File "/usr/lib/python3.10/logging/__init__.py", line 368, in getMessage
    msg = msg % self.args
TypeError: not all arguments converted during string formatting
Call stack:
  File "/usr/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/usr/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/home/ubuntu/.local/lib/python3.10/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/ubuntu/.local/lib/python3.10/site-packages/traitlets/config/application.py", line 1082, in launch_insta